# _Calibration Procedure_

Requirements prior to calibration process.
- Have at least one image with marker to mark the new datum origin.

What to expect from the calibration process.
- Calibration matrix, including both rotation and translation is saved in `.txt` format in the root directory.

This calibration process is currently limited to pitch angle (about x-axis rotation) calibration only.

### Installing all required dependencies/libraries

In [1]:
!pip install open3d
!pip install numpy
!pip install plotly
!pip install pandas
!pip install anywidget


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load uncalibrated (rotation and translation)

To visualize the image

In [13]:
import numpy as np
import open3d as o3d
import os
import plotly.graph_objects as go

# =================================================================
# PROCEDURE STEP 1: INITIAL VISUAL INSPECTION AND DATUM IDENTIFICATION
# Goal: Load the reference PCD file, visualize it, and manually record 
# the coordinates of the desired new datum point (e.g., the center of a marker).
# =================================================================

# --- CONFIGURATION ---

# 1. Full path to the single, uncalibrated reference PCD file (e.g., the one taken at the reference position).
FULL_PCD_PATH = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\uncalib\0,0,0.pcd"

# --- SCRIPT EXECUTION ---

PCD_FILENAME = os.path.basename(FULL_PCD_PATH)

if not os.path.exists(FULL_PCD_PATH):
    print(f"❌ ERROR: File not found at {FULL_PCD_PATH}")
else:
    try:
        datum_pcd = o3d.io.read_point_cloud(FULL_PCD_PATH)
        points = np.asarray(datum_pcd.points)

        print(f"Loading file: {PCD_FILENAME}")
        print("Point cloud loaded. Please use the interactive plot to find your desired datum point.")

        # --- Visualization ---
        fig = go.Figure(
            data=[
                go.Scatter3d(
                    x=points[:, 0],
                    y=points[:, 1],
                    z=points[:, 2],
                    mode='markers',
                    marker=dict(size=1, opacity=0.8, color='blue'),
                    name='Point Cloud'
                )
            ]
        )
        
        # Set the default view to the YZ plane (looking down the X-axis)
        fig.update_layout(
            title=f"STEP 1: Identify Datum Point from {PCD_FILENAME} (Uncalibrated)",
            scene=dict(
                aspectmode='data',
                xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
                camera=dict(
                    up=dict(x=0, y=0, z=1),    # Z-axis is defined as UP
                    center=dict(x=0, y=0, z=0),
                    eye=dict(x=2, y=0, z=0)    # Camera is placed on the X-axis (views YZ plane)
                )
            ),
            width=1000,
            height=800
        )
        fig.show()

        # Output instructions for the user
        print("\n--- Procedure Output ---")
        print("ACTION REQUIRED: Use the interactive plot to hover over the desired datum point.")
        print("Record the exact X, Y, Z coordinates displayed in the hover tooltip for use in the next step (Module 4).")
        print("------------------------")

    except Exception as e:
        print(f"❌ Failed to process {PCD_FILENAME}: {e}")


Loading file: 0,0,0.pcd
Point cloud loaded. Please use the interactive plot to find your desired datum point.



--- Procedure Output ---
ACTION REQUIRED: Use the interactive plot to hover over the desired datum point.
Record the exact X, Y, Z coordinates displayed in the hover tooltip for use in the next step (Module 4).
------------------------


### Applying ground levelling.

1. Apply 

In [14]:
import numpy as np
import open3d as o3d
import os
import plotly.graph_objects as go
import math

# =================================================================
# PROCEDURE STEPS 2 & 3: AUTOMATIC ROTATION AND VISUAL CONFIRMATION
# Goal: 1. Calculate the 3x3 Rotation Matrix (R) to level the ground.
#       2. Apply R, save the leveled file, and visualize the result.
#       3. Manually record the new datum coordinates (P') from the plot.
# =================================================================

# --- CONFIGURATION ---
# 1. Path to the uncalibrated file (must be the same one used in Step 1)
FULL_PCD_PATH = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\uncalib\0,0,0.pcd"

# 2. Output path for the newly rotated (leveled) file
OUTPUT_PCD_PATH = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib\rotated_0,0,0.pcd"

# 3. RANSAC PARAMETERS
GROUND_Z_THRESHOLD = 0.2   # Z-coordinate value to roughly filter points below it as 'ground'.
MAX_RANSAC_DISTANCE = 0.01  # Max distance (meters) a point can be from the RANSAC plane (e.g., 1 cm)

# --- SCRIPT EXECUTION ---

if not os.path.exists(FULL_PCD_PATH):
    print(f"❌ ERROR: Input file not found at {FULL_PCD_PATH}")
    exit()

try:
    pcd = o3d.io.read_point_cloud(FULL_PCD_PATH)
    points = np.asarray(pcd.points)
    
    # ----------------------------------------------------
    # STEP 2.1: CALCULATE 3X3 ROTATION MATRIX (R)
    # ----------------------------------------------------

    # Filter initial ground points based on Z-threshold
    ground_indices = np.where(points[:, 2] < GROUND_Z_THRESHOLD)[0]
    
    if len(ground_indices) < 100:
        print(f"⚠️ Warning: Found only {len(ground_indices)} points below Z-threshold. Adjust 'GROUND_Z_THRESHOLD'.")
        ground_pcd = pcd
    else:
        ground_pcd = pcd.select_by_index(ground_indices)
    
    # Use RANSAC to fit the best-fit plane (ground)
    plane_model, _ = ground_pcd.segment_plane(
        distance_threshold=MAX_RANSAC_DISTANCE, ransac_n=3, num_iterations=1000
    )
    
    a, b, c, d = plane_model
    normal_vector = np.array([a, b, c])
    target_vector = np.array([0, 0, 1]) # Target: Z-axis
    
    # Calculate rotation components
    rotation_axis = np.cross(normal_vector, target_vector)
    dot_product = np.clip(np.dot(normal_vector, target_vector), -1.0, 1.0)
    rotation_angle = np.arccos(dot_product)
    
    # Build the rotation matrix
    if rotation_angle == 0:
        R = np.identity(3)
    else:
        rotation_axis_normalized = rotation_axis / np.linalg.norm(rotation_axis)
        R = pcd.get_rotation_matrix_from_axis_angle(rotation_axis_normalized * rotation_angle)

    # Output the calculated rotation matrix
    print("\n--- Procedure Status ---")
    print("STEP 2.2: Calculated 3x3 Rotation Matrix (R) for Leveling:")
    print(np.round(R, 7))
    print(f"Rotation Angle Magnitude: {np.degrees(rotation_angle):.4f} degrees")
    print("------------------------")
    
    # ----------------------------------------------------
    # STEP 3.1: APPLY ROTATION AND SAVE FILE
    # ----------------------------------------------------
    
    # Apply the rotation to the point cloud data
    pcd.rotate(R, center=(0, 0, 0)) 
    rotated_points = np.asarray(pcd.points)
    
    # Ensure the output directory exists and save the file
    os.makedirs(os.path.dirname(OUTPUT_PCD_PATH), exist_ok=True)
    o3d.io.write_point_cloud(OUTPUT_PCD_PATH, pcd)
    print(f"Rotated (leveled) file saved to: {OUTPUT_PCD_PATH}")

    # ----------------------------------------------------
    # STEP 3.2: VISUALIZE ROTATED DATA
    # ----------------------------------------------------
    
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=rotated_points[:, 0],
                y=rotated_points[:, 1],
                z=rotated_points[:, 2],
                mode='markers',
                marker=dict(size=1, opacity=0.8, color='green'),
                name='Rotated Point Cloud'
            )
        ]
    )
    
    # Set the default view to the YZ plane (looking down the X-axis)
    fig.update_layout(
        title=f"STEP 3.2: Rotated Point Cloud - Confirm Leveling and New Datum Position (P')",
        scene=dict(
            aspectmode='data',
            xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
            camera=dict(
                up=dict(x=0, y=0, z=1),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=2, y=0, z=0) # Views the YZ plane
            )
        ),
        width=1000,
        height=800
    )
    fig.show()

    print("\n--- ACTION REQUIRED ---")
    print("STEP 3.3: Use the interactive plot to hover over the desired datum point (the one selected in Step 1).")
    print("Record the new X, Y, Z coordinates displayed in the hover tooltip.")
    print("These *rotated* coordinates (P') are the final coordinates before translation and will be used in Module 4.")
    print("------------------------")
    
except Exception as e:
    print(f"❌ Failed to execute ground leveling and visualization: {e}")



--- Procedure Status ---
STEP 2.2: Calculated 3x3 Rotation Matrix (R) for Leveling:
[[ 0.9996763  0.0038091 -0.0251542]
 [ 0.0038091  0.955174   0.2960205]
 [ 0.0251542 -0.2960205  0.9548503]]
Rotation Angle Magnitude: 17.2827 degrees
------------------------
Rotated (leveled) file saved to: C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib\rotated_0,0,0.pcd



--- ACTION REQUIRED ---
STEP 3.3: Use the interactive plot to hover over the desired datum point (the one selected in Step 1).
Record the new X, Y, Z coordinates displayed in the hover tooltip.
These *rotated* coordinates (P') are the final coordinates before translation and will be used in Module 4.
------------------------


### Generating 4x4 calibration matrix.

1. Replace the rotation matrix in the script below with the one generated from `1. Apply` from `Applying Ground Levelling`
2. From the same image, appoint one point cloud to be assign as the new datum. Replace the value in the script below as P'

*either of the script below doing the same thing, generating and saved calibration matrix in `.txt` extension named `final_calibration_matrix_T.txt`, suggested for the later as the source and destination files is known*

In [19]:
import numpy as np

# =================================================================
# PROCEDURE STEP 4: CALCULATE FINAL 4X4 CALIBRATION MATRIX (T)
# Goal: Combine the 3x3 rotation matrix (R) and the negative of the 
# rotated datum point (P') into the final 4x4 transformation matrix (T).
# Translation Vector t = -P'
# =================================================================

# --- CONFIGURATION (ACTION REQUIRED) ---

# 1. Manually copy the 3x3 rotation matrix (R) from Module 2/3 output here:
R = np.array([
    [0.9997094, 0.0035611, -0.0238409],
    [0.0035611,  0.9563582,  0.2921751],
    [0.0238409, -0.2921751,  0.9560676]
]) # <== REPLACE WITH YOUR CALCULATED R MATRIX

# 2. Manually copy the datum point coordinates *after* rotation (P') from Module 3 visualization here:
# This point is what should be moved to the origin (0, 0, 0).
P_prime_rotated = np.array([-0.0171737, 2.09079, -1.237612]) # <== REPLACE WITH THE MANUALLY SELECTED P' POINT

# --- SCRIPT EXECUTION ---

print("\n--- Starting Calculation of 4x4 Matrix ---")

# 1. Calculate the translation vector t
# The translation vector is the negative of the rotated datum point coordinates.
translation_vector_t = -P_prime_rotated

# 2. Construct the 4x4 Transformation Matrix (T)
# The structure is: T = | R  | t |
#                      | 000| 1 |
FINAL_CALIBRATION_MATRIX_4X4 = np.identity(4)
FINAL_CALIBRATION_MATRIX_4X4[:3, :3] = R                  # Insert the 3x3 Rotation
FINAL_CALIBRATION_MATRIX_4X4[:3, 3] = translation_vector_t # Insert the Translation Vector

# 3. Output the final matrix
print("\n--- Procedure Output ---")
print("STEP 4: FINAL 4x4 CALIBRATION MATRIX (T):")
print(np.round(FINAL_CALIBRATION_MATRIX_4X4, 7))
print("\nTranslation Vector (t):")
print(np.round(translation_vector_t, 7))
print("------------------------")

# Save the final matrix for permanent record and use in the last module
np.savetxt('final_calibration_matrix_T.txt', FINAL_CALIBRATION_MATRIX_4X4)
print("\nFinal 4x4 matrix saved to 'final_calibration_matrix_T.txt'")

# Visualization setup
hover_text = (
    f"<b>Datum Point (Calibrated)</b><br>"
    f"Final Coords: ({np.round(datum_point_calibrated, 7)})<br>"
    f"Expected: (0, 0, 0)"
)

trace_calibrated = go.Scatter3d(
    x=points_calibrated[:, 0], y=points_calibrated[:, 1], z=points_calibrated[:, 2],
    mode='markers', marker=dict(size=1.5, color='blue', opacity=0.8),
    name='Calibrated Cloud'
)

# Highlight the new datum point
trace_datum = go.Scatter3d(
    x=[datum_point_calibrated[0]], y=[datum_point_calibrated[1]], z=[datum_point_calibrated[2]],
    mode='markers', marker=dict(size=8, color='red'), name='New Datum Point (Origin)',
    text=[hover_text], hoverinfo="text"
)

fig = go.Figure(data=[trace_calibrated, trace_datum])

# Set the default view to the YZ plane (looking down the X-axis)
fig.update_layout(
    title="STEP 5.3: Verification - Fully Calibrated Point Cloud (Datum Point in Red)",
    scene=dict(
        aspectmode='data',
        xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
        camera=dict(
            up=dict(x=0, y=0, z=1),  # Z-axis is defined as UP
            center=dict(x=0, y=0, z=0),
            eye=dict(x=2, y=0, z=0)  # Camera is placed on the X-axis (views YZ plane)
        )
    ),
    width=900, height=700
)
fig.show()


--- Starting Calculation of 4x4 Matrix ---

--- Procedure Output ---
STEP 4: FINAL 4x4 CALIBRATION MATRIX (T):
[[ 0.9997094  0.0035611 -0.0238409  0.0171737]
 [ 0.0035611  0.9563582  0.2921751 -2.09079  ]
 [ 0.0238409 -0.2921751  0.9560676  1.237612 ]
 [ 0.         0.         0.         1.       ]]

Translation Vector (t):
[ 0.0171737 -2.09079    1.237612 ]
------------------------

Final 4x4 matrix saved to 'final_calibration_matrix_T.txt'


In [21]:
import numpy as np
import open3d as o3d
import os
import plotly.graph_objects as go

# =================================================================
# PROCEDURE STEP 4 & 5: CALCULATE 4X4 MATRIX, APPLY, AND VERIFY VISUALLY
# Goal: 1. Assemble the final 4x4 matrix (T).
#       2. Apply T to the original file to get the final calibrated result.
#       3. Visualize the result to confirm leveling and that the new datum is at (0, 0, 0).
# =================================================================

# --- CONFIGURATION (YOUR PROVIDED DATA) ---

# 1. Path to the ORIGINAL, UNCALIBRATED file used for calibration.
TARGET_FILE_PATH_ORIGINAL = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\uncalib\0,0,0.pcd"

# 2. Path to save the final calibrated file.
OUTPUT_FILE_PATH_CALIBRATED = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib\full-calibrated.pcd"

# 3. Manually copy the 3x3 rotation matrix (R) from Module 2/3 output here:
R = np.array([
    [0.9997094, 0.0035611, -0.0238409],
    [0.0035611, 0.9563582, 0.2921751],
    [0.0238409, -0.2921751, 0.9560676]
])

# 4. Manually copy the datum point coordinates *after* rotation (P') from Module 3 visualization:
P_prime_rotated = np.array([-0.0171737, 2.09079, -1.237612]) 

# 5. Manually copy the *original* coordinates of the datum point (P) from Module 1:
ORIGINAL_DATUM_COORDS = np.array([-0.04256823, 2.371565, -0.5614936]) 

# --- SCRIPT EXECUTION ---

if not os.path.exists(TARGET_FILE_PATH_ORIGINAL):
    print(f"❌ ERROR: Original file not found at {TARGET_FILE_PATH_ORIGINAL}")
    exit()

try:
    # Load the original file
    pcd_original = o3d.io.read_point_cloud(TARGET_FILE_PATH_ORIGINAL)
    
    # 🌟 CRITICAL STEP: Store the points BEFORE transformation to find the correct index.
    points_original = np.asarray(pcd_original.points).copy()

    # ----------------------------------------------------
    # STEP 4.1: CALCULATE FINAL 4X4 MATRIX (T)
    # ----------------------------------------------------
    
    translation_vector_t = -P_prime_rotated
    FINAL_CALIBRATION_MATRIX_4X4 = np.identity(4)
    FINAL_CALIBRATION_MATRIX_4X4[:3, :3] = R 
    FINAL_CALIBRATION_MATRIX_4X4[:3, 3] = translation_vector_t
    
    print("\n--- Procedure Output ---")
    print("STEP 4.2: FINAL 4x4 CALIBRATION MATRIX (T) Calculated:")
    print(np.round(FINAL_CALIBRATION_MATRIX_4X4, 7))
    print("------------------------")
    
    np.savetxt('final_calibration_matrix_T.txt', FINAL_CALIBRATION_MATRIX_4X4)
    print("Final 4x4 matrix saved to 'final_calibration_matrix_T.txt'")


    # ----------------------------------------------------
    # STEP 5.1: APPLY AND SAVE THE CALIBRATION
    # ----------------------------------------------------
    
    # Apply the full 4x4 transformation to the original point cloud
    pcd_original.transform(FINAL_CALIBRATION_MATRIX_4X4)
    pcd_calibrated = pcd_original
    
    # Save the fully calibrated file
    os.makedirs(os.path.dirname(OUTPUT_FILE_PATH_CALIBRATED), exist_ok=True)
    o3d.io.write_point_cloud(OUTPUT_FILE_PATH_CALIBRATED, pcd_calibrated)
    print(f"\nFully Calibrated file saved to: {OUTPUT_FILE_PATH_CALIBRATED}")


    # ----------------------------------------------------
    # STEP 5.2: VERIFICATION AND VISUALIZATION (ROBUST INDEXING)
    # ----------------------------------------------------
    points_calibrated = np.asarray(pcd_calibrated.points)

    # 1. Find the index of the point closest to the ORIGINAL datum coordinates
    distances_to_original_datum = np.linalg.norm(points_original - ORIGINAL_DATUM_COORDS, axis=1)
    datum_point_idx = np.argmin(distances_to_original_datum)
    
    # 2. Use that index to get the coordinates from the final calibrated cloud
    datum_point_calibrated = points_calibrated[datum_point_idx, :]
    
    # Print comparison
    print("\n--- Verification Summary ---")
    print(f"Located Datum Point (P''): {np.round(datum_point_calibrated, 7)}")
    print(f"Distance from Origin (Residual Error): {np.linalg.norm(datum_point_calibrated):.7f} m")
    print("----------------------------")
    
    # Visualization setup
    hover_text_located = (
        f"<b>Located Datum Point (P'')</b><br>"
        f"Final Coords: ({np.round(datum_point_calibrated, 7)})<br>"
        f"Error: {np.linalg.norm(datum_point_calibrated):.7f} m"
    )
    hover_text_target = "<b>Theoretical Target (0, 0, 0)</b>"
    
    trace_calibrated = go.Scatter3d(
        x=points_calibrated[:, 0], y=points_calibrated[:, 1], z=points_calibrated[:, 2],
        mode='markers', marker=dict(size=1.5, color='blue', opacity=0.8),
        name='Calibrated Cloud'
    )
    
    # Trace 1: Highlight the actual LOCATED datum point (Red)
    trace_datum_located = go.Scatter3d(
        x=[datum_point_calibrated[0]], y=[datum_point_calibrated[1]], z=[datum_point_calibrated[2]],
        mode='markers', marker=dict(size=8, color='red'), name='Located Datum Point (P\'\')',
        text=[hover_text_located], hoverinfo="text"
    )
    
    # Trace 2: Highlight the THEORETICAL target origin (Green Cross)
    trace_datum_target = go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers', marker=dict(size=12, color='green', symbol='cross'), name='Theoretical Target (0, 0, 0)',
        text=[hover_text_target], hoverinfo="text"
    )


    fig = go.Figure(data=[trace_calibrated, trace_datum_located, trace_datum_target])

    # Set the default view to the YZ plane (looking down the X-axis)
    fig.update_layout(
        title="STEP 5.3: Verification - Fully Calibrated Point Cloud (Red=Located, Green=Target)",
        scene=dict(
            aspectmode='data',
            xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
            camera=dict(
                up=dict(x=0, y=0, z=1),  # Z-axis is defined as UP
                center=dict(x=0, y=0, z=0),
                eye=dict(x=2, y=0, z=0)  # Camera is placed on the X-axis (views YZ plane)
            )
        ),
        width=900, height=700
    )
    fig.show()

except Exception as e:
    print(f"❌ Failed to run final calibration and verification: {e}")



--- Procedure Output ---
STEP 4.2: FINAL 4x4 CALIBRATION MATRIX (T) Calculated:
[[ 0.9997094  0.0035611 -0.0238409  0.0171737]
 [ 0.0035611  0.9563582  0.2921751 -2.09079  ]
 [ 0.0238409 -0.2921751  0.9560676  1.237612 ]
 [ 0.         0.         0.         1.       ]]
------------------------
Final 4x4 matrix saved to 'final_calibration_matrix_T.txt'

Fully Calibrated file saved to: C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib\full-calibrated.pcd

--- Verification Summary ---
Located Datum Point (P''): [0.e+00 2.e-07 3.e-07]
Distance from Origin (Residual Error): 0.0000003 m
----------------------------


### Batch calibration 

1. This is the part where we apply the calibration matrix that had been generated and saved in .txt form to all the files that comes from the same sensor.
2. Select the folder where all the uncalibrated data files and assign it to `INPUT_DIR`.
3. Assign the output directory to `OUTPUT_DIR` variable.

In [22]:
import numpy as np
import open3d as o3d
import os
import glob

# =================================================================
# MODULE 6: BATCH APPLICATION OF FINAL CALIBRATION MATRIX
# Goal: Apply the FINAL 4x4 matrix (T) to all uncalibrated files.
# =================================================================

# --- CONFIGURATION (ACTION REQUIRED) ---

# 1. Path to the folder containing your FINAL 4x4 matrix (T) file.
#    This file must be named 'final_calibration_matrix_T.txt'
#    (It should be in the same folder as this script if you ran the last module.)
MATRIX_FILE_PATH = 'final_calibration_matrix_T.txt'

# 2. Path to the directory containing ALL UNCALIBRATED POINT CLOUDS.
INPUT_DIR = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\uncalib"

# 3. Path to the output directory where ALL CALIBRATED files will be saved.
#    The script will create this folder and mirror the structure from INPUT_DIR.
OUTPUT_DIR = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib_batch"


# --- SCRIPT EXECUTION ---

print("--- Starting Batch Calibration Application (Module 6) ---")

# 1. Load the final 4x4 transformation matrix (T)
try:
    T_matrix = np.loadtxt(MATRIX_FILE_PATH)
    if T_matrix.shape != (4, 4):
        raise ValueError("Matrix loaded is not a 4x4 matrix.")
    print(f"\n✅ Successfully loaded 4x4 transformation matrix from '{MATRIX_FILE_PATH}'.")
    print("Matrix T:\n", np.round(T_matrix, 5))
except Exception as e:
    print(f"\n❌ ERROR: Could not load or verify the transformation matrix: {e}")
    print(f"Please ensure '{MATRIX_FILE_PATH}' exists and contains a valid 4x4 numpy array.")
    exit()

# 2. Find all PCD files in the input directory (recursively)
# Searches all subdirectories (**) for files ending in .pcd
pcd_files = glob.glob(os.path.join(INPUT_DIR, '**', '*.pcd'), recursive=True)

if not pcd_files:
    print(f"\n⚠️ WARNING: No .pcd files found in input directory: {INPUT_DIR}. Exiting.")
    exit()

print(f"\nFound {len(pcd_files)} files to process. Starting transformation...")

# 3. Process each file
processed_count = 0
for file_path_input in pcd_files:
    try:
        # Determine the relative path to preserve the original folder structure
        relative_path = os.path.relpath(file_path_input, INPUT_DIR)
        file_path_output = os.path.join(OUTPUT_DIR, relative_path)
        
        # Ensure the output directory structure exists
        os.makedirs(os.path.dirname(file_path_output), exist_ok=True)

        print(f"-> Processing: {relative_path}")
        
        # Load, transform, and save
        pcd = o3d.io.read_point_cloud(file_path_input)
        pcd.transform(T_matrix)
        o3d.io.write_point_cloud(file_path_output, pcd)
        
        processed_count += 1
        
    except Exception as e:
        print(f"   ❌ Failed to process {relative_path}: {e}")
        continue # Move to the next file if one fails

print(f"\n--- Batch Application Complete ---")
print(f"Successfully calibrated and saved {processed_count} files to: {OUTPUT_DIR}")
print("----------------------------------")


--- Starting Batch Calibration Application (Module 6) ---

✅ Successfully loaded 4x4 transformation matrix from 'final_calibration_matrix_T.txt'.
Matrix T:
 [[ 0.99971  0.00356 -0.02384  0.01717]
 [ 0.00356  0.95636  0.29218 -2.09079]
 [ 0.02384 -0.29218  0.95607  1.23761]
 [ 0.       0.       0.       1.     ]]

Found 4 files to process. Starting transformation...
-> Processing: 0,0,0.pcd
-> Processing: 0,1.2,0.pcd
-> Processing: 0.9,0,0.pcd
-> Processing: 0.9,1.2,0.pcd

--- Batch Application Complete ---
Successfully calibrated and saved 4 files to: C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib_batch
----------------------------------


# COMPARISON VIEWER
(keep this at the lowest)

1. This algorithm is used just to visualize the uncalibrated and calibrated file. Replace both files path accordingly to visualize it.

In [23]:
import numpy as np
import open3d as o3d
import os
import plotly.graph_objects as go

# --- CONFIGURATION ---

# IMPORTANT: Enter the full path to the ORIGINAL, UNCALIBRATED file.
TARGET_FILE_PATH_ORIGINAL = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\uncalib\0,0,0.pcd"

# IMPORTANT: Enter the full path to the CALIBRATED file.
TARGET_FILE_PATH_CALIBRATED = r"C:\Users\Hazimi\OneDrive\CEEM222 6M MEC698 Internship\ground_calibration\calib_batch\0,0,0.pcd"

# A small tolerance to consider a point "at the origin"
ORIGIN_TOLERANCE = 1e-4

# --- MAIN SCRIPT ---

if os.path.exists(TARGET_FILE_PATH_ORIGINAL) and os.path.exists(TARGET_FILE_PATH_CALIBRATED):
    print("✅ Found both original and calibrated files.")
    try:
        # Load both point clouds
        pcd_original = o3d.io.read_point_cloud(TARGET_FILE_PATH_ORIGINAL)
        pcd_calibrated = o3d.io.read_point_cloud(TARGET_FILE_PATH_CALIBRATED)

        points_original = np.asarray(pcd_original.points)
        points_calibrated = np.asarray(pcd_calibrated.points)

        # Check for empty point clouds
        if points_original.size == 0 or points_calibrated.size == 0:
            raise ValueError("One of the point clouds is empty.")

        # --- FINDING THE DATUM POINT ---
        # Find the point closest to the origin in the CALIBRATED point cloud
        distances_to_origin = np.linalg.norm(points_calibrated, axis=1)
        datum_point_idx = np.argmin(distances_to_origin)
        
        # Get the coordinates of that same point in both clouds
        datum_point_calibrated = points_calibrated[datum_point_idx, :]
        datum_point_original = points_original[datum_point_idx, :]

        # --- VISUALIZATION ---

        # Create two traces for the point clouds
        trace_original = go.Scatter3d(
            x=points_original[:, 0],
            y=points_original[:, 1],
            z=points_original[:, 2],
            mode='markers',
            marker=dict(size=1.5, color='gray', opacity=0.8),
            name='Original Point Cloud'
        )

        trace_calibrated = go.Scatter3d(
            x=points_calibrated[:, 0],
            y=points_calibrated[:, 1],
            z=points_calibrated[:, 2],
            mode='markers',
            marker=dict(size=1.5, color='blue', opacity=0.8),
            name='Calibrated Point Cloud'
        )

        # Create a separate trace for the datum point with custom hover text
        hover_text = (
            f"<b>Datum Point</b><br>"
            f"Original Coords: ({np.round(datum_point_original[0], 4)}, "
            f"{np.round(datum_point_original[1], 4)}, "
            f"{np.round(datum_point_original[2], 4)})<br>"
            f"New Coords: ({np.round(datum_point_calibrated[0], 4)}, "
            f"{np.round(datum_point_calibrated[1], 4)}, "
            f"{np.round(datum_point_calibrated[2], 4)})"
        )
        
        trace_datum = go.Scatter3d(
            x=[datum_point_calibrated[0]],
            y=[datum_point_calibrated[1]],
            z=[datum_point_calibrated[2]],
            mode='markers',
            marker=dict(size=8, color='red'),
            name='New Datum Point',
            text=[hover_text],
            hoverinfo="text"
        )

        # Create the figure with all traces
        fig = go.Figure(data=[trace_original, trace_calibrated, trace_datum])

        # Update layout for a better view
        fig.update_layout(
            title="Verification of Calibration Matrix",
            scene=dict(
                aspectmode='data',
                xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
                camera=dict(
                    up=dict(x=0, y=0, z=1),
                    eye=dict(x=1.5, y=0, z=0)
                )
            ),
            showlegend=True,
            width=900,
            height=700
        )

        fig.show()

    except Exception as e:
        print(f"❌ Error processing the file: {e}")
        print("The files may be corrupted or not a valid .pcd format.")
else:
    print(f"❌ Error: One or both of the files were not found.")
    print("Please check the paths and filenames for any typos.")


✅ Found both original and calibrated files.
